<a href="https://colab.research.google.com/github/khushipatil05/ontology-aware-protein-function-prediction/blob/main/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Go to Runtime > Change runtime type > T4 GPU, then restart and run again.")

CUDA available: True
GPU: Tesla T4


In [17]:
GITHUB_REPO = "https://github.com/khushipatil05/ontology-aware-protein-function-prediction.git"
DRIVE_ROOT  = "/content/drive/MyDrive"
REPO_NAME   = "ontology-aware-protein-function-prediction"
EMBED_LIMIT = None      # None = all proteins
LAMBDAS     = [0.0, 0.1, 0.5, 1.0] # 0.0 = baseline (no hierarchy loss), 0.5 = ontology-aware

In [18]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
PROJECT = f"{DRIVE_ROOT}/{REPO_NAME}"
os.makedirs(PROJECT, exist_ok=True)

if os.path.exists(f"{PROJECT}/src/config.py"):
    # code already there (cloned before, or uploaded manually) -> just update if it is a git repo
    if os.path.exists(f"{PROJECT}/.git"):
        print(subprocess.run(["git", "-C", PROJECT, "pull"], capture_output=True, text=True).stdout)
    print("Code found at", PROJECT)
else:
    # PROJECT may already contain data/processed.zip, so clone into a temp dir and copy over
    subprocess.run(["rm", "-rf", "/content/_tmp_repo"])
    r = subprocess.run(["git", "clone", GITHUB_REPO, "/content/_tmp_repo"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    subprocess.run("cp -r /content/_tmp_repo/. " + PROJECT + "/", shell=True)
    assert os.path.exists(f"{PROJECT}/src/config.py"), (
        "src/config.py not found. Push your code to GitHub first, or upload the repo folder to Drive.")

%cd {PROJECT}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Updating 237a71e..bde345e
Fast-forward
 notebooks/01_data_preprocessing.ipynb | 549 +++++++++++++++++-----------------
 1 file changed, 275 insertions(+), 274 deletions(-)

Code found at /content/drive/MyDrive/ontology-aware-protein-function-prediction
/content/drive/MyDrive/ontology-aware-protein-function-prediction


In [ ]:
!pip install -q transformers scikit-learn scipy tqdm


In [19]:
import os, json, zipfile, gzip, shutil
proc = "data/processed"
os.makedirs(proc, exist_ok=True)

# 1. Unzip data/processed.zip
if not os.path.exists(f"{proc}/terms.tsv"):
    z = "data/processed.zip"
    assert os.path.exists(z), "Upload processed.zip to " + os.path.abspath("data") + " (see guide, Step 3)"
    with zipfile.ZipFile(z) as f:
        for m in f.namelist():
            name = m.split("processed/", 1)[1] if "processed/" in m else m
            if name and not name.endswith("/"):
                out_path = f"{proc}/{name}"
                os.makedirs(os.path.dirname(out_path) or proc, exist_ok=True)
                with open(out_path, "wb") as out:
                    out.write(f.read(m))

# 2. Decompress proteins.tsv.gz if it exists
gz_path = f"{proc}/proteins.tsv.gz"
if os.path.exists(gz_path) and not os.path.exists(f"{proc}/proteins.tsv"):
    print("Decompressing proteins.tsv.gz...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(f"{proc}/proteins.tsv", "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Decompression complete.")

# 3. Final verification of required files
for need in ["proteins.tsv", "terms.tsv", "labels.npz", "hierarchy.npy", "stats.json"]:
    assert os.path.exists(f"{proc}/{need}"), f"Missing {proc}/{need}"
print(json.dumps(json.load(open(f"{proc}/stats.json")), indent=2))

{
  "n_proteins": 466802,
  "n_terms": 4181,
  "n_edges": 29525,
  "train": 373441,
  "val": 46680,
  "test": 46681
}


In [20]:
limit = f"--limit {EMBED_LIMIT}" if EMBED_LIMIT else ""
!python src/embed.py {limit} --token_budget 16000

Loading weights: 100% 534/534 [00:00<00:00, 896.69it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
shards: 100% 94/94 [00:00<00:00, 1788.11it/s]
Saved (466802, 1280)


In [21]:
!ls -lh data/processed/

total 1.4G
-rw------- 1 root root  136 Oct 10 13:09 embedded_n.npy
drwx------ 2 root root 4.0K Oct  5 15:53 embeddings
-rw------- 1 root root 1.2G Oct 10 13:09 embeddings.npy
-rw------- 1 root root 462K Oct  5 15:41 hierarchy.npy
-rw------- 1 root root  13M Oct  5 15:30 labels.npz
-rw------- 1 root root 172M Oct  5 15:30 proteins.tsv
-rw------- 1 root root  96M Oct  5 15:30 proteins.tsv.gz
-rw------- 1 root root  124 Oct  5 15:41 stats.json
-rw------- 1 root root 231K Oct  5 15:41 terms.tsv


In [7]:
!file data/processed/proteins.tsv

data/processed/proteins.tsv: ASCII text, with very long lines (1044), with CRLF line terminators


In [8]:
!file data/processed/proteins.tsv.gz

data/processed/proteins.tsv.gz: gzip compressed data, was "proteins.tsv", last modified: Mon Oct  5 13:10:02 2026, from Unix, original size modulo 2^32 179718569


In [9]:
with open("data/processed/proteins.tsv", "rb") as f:
    print(f.read(10))

b'accession\t'


In [10]:
with open("data/processed/proteins.tsv.gz", "rb") as f:
    print(f.read(10))

b'\x1f\x8b\x08\x08\xaa\xa1\xc3j\x00\x03'


In [11]:
!rm data/processed/proteins.tsv

In [12]:
!gzip -dk data/processed/proteins.tsv.gz

In [13]:
!file data/processed/proteins.tsv

data/processed/proteins.tsv: ASCII text, with very long lines (1044), with CRLF line terminators


In [22]:
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    print("="*20, tag, "="*20)
    !python src/train.py --lam {lam} --tag {tag}

==================== baseline ====================
epoch 01  train 0.0226  val 0.0133  (bce 0.0133 hier 0.0000)
epoch 02  train 0.0130  val 0.0112  (bce 0.0112 hier 0.0000)
epoch 03  train 0.0116  val 0.0104  (bce 0.0104 hier 0.0000)
epoch 04  train 0.0109  val 0.0098  (bce 0.0098 hier 0.0000)
epoch 05  train 0.0104  val 0.0095  (bce 0.0095 hier 0.0000)
epoch 06  train 0.0100  val 0.0092  (bce 0.0092 hier 0.0000)
epoch 07  train 0.0097  val 0.0090  (bce 0.0090 hier 0.0000)
epoch 08  train 0.0095  val 0.0089  (bce 0.0089 hier 0.0000)
epoch 09  train 0.0093  val 0.0087  (bce 0.0087 hier 0.0000)
epoch 10  train 0.0092  val 0.0086  (bce 0.0086 hier 0.0000)
epoch 11  train 0.0090  val 0.0085  (bce 0.0085 hier 0.0000)
epoch 12  train 0.0089  val 0.0084  (bce 0.0084 hier 0.0000)
epoch 13  train 0.0088  val 0.0084  (bce 0.0084 hier 0.0000)
epoch 14  train 0.0087  val 0.0083  (bce 0.0083 hier 0.0000)
epoch 15  train 0.0086  val 0.0082  (bce 0.0082 hier 0.0000)
epoch 16  train 0.0085  val 0.0082

In [23]:
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    print("="*20, tag, "="*20)
    !python src/evaluate.py --tag {tag}

==================== baseline ====================
/content/drive/MyDrive/ontology-aware-protein-function-prediction/src/model.py:38: UserWarning: index_reduce() is in beta and the API may change at any time. (Triggered internally at /pytorch/aten/src/ATen/native/TensorAdvancedIndexing.cpp:1513.)
  out.index_reduce_(1, parent_idx, probs[:, child_idx], "amax", include_self=True)
Molecular Function {
 "raw": {
  "Fmax": 0.9254,
  "threshold": 0.4,
  "precision_micro": 0.9412,
  "recall_micro": 0.9289,
  "F1_micro": 0.935,
  "AUPR_micro": 0.9768,
  "AUPR_macro": 0.902
 },
 "hierarchy_enforced": {
  "Fmax": 0.9295,
  "threshold": 0.48,
  "precision_micro": 0.9473,
  "recall_micro": 0.9287,
  "F1_micro": 0.9379,
  "AUPR_micro": 0.9781,
  "AUPR_macro": 0.9032
 },
 "violation_rate_raw@0.5": 0.0132,
 "violation_rate_enforced@0.5": 0.0
}
Biological Process {
 "raw": {
  "Fmax": 0.8913,
  "threshold": 0.44,
  "precision_micro": 0.9028,
  "recall_micro": 0.7865,
  "F1_micro": 0.8407,
  "AUPR_micr

In [16]:
import json, pandas as pd
rows = []
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    res = json.load(open(f"results/{tag}_test_metrics.json"))
    for ont, r in res.items():
        rows.append({"model": tag, "ontology": ont,
                     "Fmax": r["raw"]["Fmax"], "AUPR_micro": r["raw"]["AUPR_micro"],
                     "AUPR_macro": r["raw"]["AUPR_macro"],
                     "violations@0.5 (raw)": r["violation_rate_raw@0.5"],
                     "violations@0.5 (enforced)": r["violation_rate_enforced@0.5"]})
pd.DataFrame(rows)

,model,ontology,Fmax,AUPR_micro,AUPR_macro,violations@0.5 (raw),violations@0.5 (enforced)
0,baseline,Molecular Function,0.9254,0.9768,0.9020,0.0132,0.0
1,baseline,Biological Process,0.8913,0.9055,0.7059,0.0132,0.0
2,baseline,Cellular Component,0.9089,0.9222,0.6867,0.0132,0.0
3,mlp_lam0.5,Molecular Function,0.8521,0.9156,0.7265,0.0123,0.0
4,mlp_lam0.5,Biological Process,0.8099,0.7876,0.5008,0.0123,0.0
5,mlp_lam0.5,Cellular Component,0.8552,0.8126,0.3370,0.0123,0.0
